# 01 — Dataset Audit

Purpose: Inventory NASA, CALCE, and Oxford cells, cycles, conditions, missing channels, exclusions, and provenance.

Leakage rule: current-cycle full discharged capacity and SOH are labels only.
Preprocessing is fitted on training batteries, validation batteries control
selection and early stopping, and test batteries are scored exactly once.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "v3"
RESULTS = ARTIFACT_ROOT / "results"
FIGURES = ARTIFACT_ROOT / "figures"
FEATURES = ARTIFACT_ROOT / "features"
for path in (RESULTS, FIGURES, FEATURES):
    path.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")


Project root: E:\VIT\aiBatteryLifecycle\application


## Normalize raw cycles

This step engineers no predictors. Every exclusion is recorded with a reason.

In [2]:
import pandas as pd
from scripts.data.build_benchmark_datasets import load_all_cycles
from src.data.adapters import cycle_inventory

cycles_by_dataset = load_all_cycles(PROJECT_ROOT / "datasets" / "raw")
inventories = {}
for name, cycles in cycles_by_dataset.items():
    inventory = cycle_inventory(cycles)
    inventory.to_csv(FEATURES / f"{name.lower()}_inventory.csv", index=False)
    inventories[name] = inventory
    display(inventory.groupby("battery_id").agg(
        cycles=("cycle_number", "count"),
        capacity_min_ah=("capacity_ah", "min"),
        capacity_max_ah=("capacity_ah", "max"),
    ))
audit = pd.concat(inventories.values(), ignore_index=True)
display(audit.groupby("dataset").agg(
    batteries=("battery_id", "nunique"),
    cycles=("cycle_number", "count"),
    measurements=("n_measurements", "sum"),
))

,cycles,capacity_min_ah,capacity_max_ah
battery_id,,,
B0005,168,1.287453,1.856487
B0006,168,1.153818,2.035338
B0007,168,1.400455,1.891052
B0018,132,1.341051,1.855005
B0025,56,1.767789,1.848984
B0026,56,1.386337,1.816528
B0027,56,1.770093,1.823308
B0028,56,1.717234,1.804783
B0029,40,1.612080,1.844701


,cycles,capacity_min_ah,capacity_max_ah
battery_id,,,
CS2_35,878,0.303643,1.138460
CS2_36,909,0.303045,1.144814
CS2_37,986,0.313383,1.134949
CS2_38,1016,0.300230,1.139524


,cycles,capacity_min_ah,capacity_max_ah
battery_id,,,
Cell1,78,0.560014,0.739111
Cell2,73,0.501435,0.734819
Cell3,76,0.565857,0.735120
Cell4,47,0.581234,0.734076
Cell5,46,0.456129,0.735336
Cell6,46,0.579023,0.732795
Cell7,77,0.582807,0.728746
Cell8,76,0.558161,0.727528


,batteries,cycles,measurements
dataset,,,
CALCE,4,3789,364787
NASA,34,2881,830026
Oxford,8,519,1610378
